# 03 — Feature Engineering

Run bronze + silver ETL, inspect lag/rolling feature distributions,
validate point-in-time correctness, and visualize feature-target correlations.

**Prerequisites:** Raw data collected (NB01).

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 3.1 Build bronze layer

In [ ]:
from etl.bronze import build_bronze
build_bronze()

## 3.2 Build silver layer (features)

In [ ]:
from etl.silver import build_silver
build_silver()

## 3.3 Inspect silver schema

In [ ]:
con = duckdb.connect()
SILVER = '../data/silver/features.parquet'

schema = con.execute(f'DESCRIBE SELECT * FROM read_parquet("{SILVER}")').fetchdf()
print(f'{len(schema)} columns:')
for _, row in schema.iterrows():
    print(f'  {row.column_name:<35} {row.column_type}')

## 3.4 Validate point-in-time correctness

At row `t`, features should only use data from `t-N` through `t-1`. We verify:
- `price_lag_1h` at row `t` == `price_eur_mwh` at row `t-1`
- No current-row value should appear in rolling mean

In [ ]:
df = con.execute(f"SELECT * FROM read_parquet('{SILVER}') ORDER BY timestamp_utc LIMIT 200").fetchdf()

# Check lag correctness: lag_1h[i] should equal price[i-1]
match = (df['price_lag_1h'].iloc[1:].values == df['price_eur_mwh'].iloc[:-1].values)
print(f'price_lag_1h == price[t-1]: {match.mean():.1%} match (should be ~100%)')

## 3.5 Feature-target correlation (per horizon)

In [ ]:
from etl.gold import build_gold
build_gold()

In [ ]:
from ml.train import FEATURE_COLS, TARGET_COLS

ABT = '../data/gold/abt_train.parquet'
df_abt = con.execute(f'SELECT * FROM read_parquet("{ABT}")').fetchdf()

# Correlation of each feature with price_t_plus_1h and price_t_plus_24h
corr_1h  = df_abt[FEATURE_COLS + ['price_t_plus_1h']].corr()['price_t_plus_1h'].drop('price_t_plus_1h')
corr_24h = df_abt[FEATURE_COLS + ['price_t_plus_24h']].corr()['price_t_plus_24h'].drop('price_t_plus_24h')

df_corr = pd.DataFrame({'t+1h': corr_1h, 't+24h': corr_24h}).sort_values('t+1h', ascending=False)

fig = go.Figure()
fig.add_bar(x=df_corr.index, y=df_corr['t+1h'], name='t+1h')
fig.add_bar(x=df_corr.index, y=df_corr['t+24h'], name='t+24h')
fig.update_layout(title='Feature-Target Correlation (t+1h vs t+24h)',
                  template='plotly_dark', barmode='group', height=450,
                  xaxis_tickangle=-45)
fig.show()